ACROSS-PARTICIPANT RSA

In [ ]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr, zscore, ttest_rel, pearsonr, ttest_ind
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
import seaborn as sns
from scipy.stats import sem
from scipy.signal import detrend
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from scipy.stats import ttest_1samp
from sklearn.model_selection import KFold
from tqdm import trange
import pandas as pd
import statsmodels.formula.api as smf
from tqdm import tqdm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
foundationScores_dir = "/Users/silvycollin/Documents/GitHub/wholeBrain_narratives_MAC_MFT/text/foundationScores/"

In [ ]:
whichVersion = 'persentence' #persentence, 3sec, 6sec

whichFoundationScore = "MAC_a_moral_nonmoral"

# save
save_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/RSA_relToMoralnonmoral/"
)

In [ ]:
# ------------------------
# Load 
# ------------------------

whichROI = "TPJ_refined_mask" # hippocampus,STG_MTG_refined_mask,TPJ_refined_mask

whichStory = "tunnel"  # tunnel, 21styear

base_dir = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_first_level_per_sentence/"
)

if whichVersion =='persentence':
    foundationScores_file = f"{foundationScores_dir}{whichStory}_MFT_MAC.xlsx"
    foundationScores = pd.read_excel(foundationScores_file)

    # Construct the filename
    file_path = str(save_dir) + f"{whichVersion}_{whichStory}_RDM_fast_embeddings.csv"
    
    # Load it back as a pandas DataFrame
    RDM_df = pd.read_csv(file_path, index_col=0)  # index_col=0 if you saved the index
    
elif whichVersion =='3sec':
    print('3')
elif whichVersion =='6sec':
    print('6')
    
print(whichStory)
print(whichROI)

data_dir = f"{base_dir}{whichStory}/"

left = np.load(data_dir + f"{whichStory}_{whichROI}_L_3D_clean.npy")
right = np.load(data_dir + f"{whichStory}_{whichROI}_R_3D_clean.npy")

In [ ]:
# ------------------------
# load foundation scores 
# ------------------------
# start and end column is start and end of the sentences in seconds, TR was 1.5 sec

foundationScoresSelect = foundationScores[["MAC_a_moral_nonmoral","start","end"]]
foundationScoresSelect = foundationScoresSelect.copy()

# go from seconds to TRs for start and end column
foundationScoresSelect["startTR"] = foundationScoresSelect["start"] / 1.5
foundationScoresSelect["endTR"] = foundationScoresSelect["end"] / 1.5

# round to full TRs/volumes
foundationScoresSelect["startTR"] = foundationScoresSelect["startTR"].round(0).astype(int)
foundationScoresSelect["endTR"] = foundationScoresSelect["endTR"].round(0).astype(int)

# shift by 3 TRs for the HRF
foundationScoresSelect['startTR'] = foundationScoresSelect['startTR'] + 3
foundationScoresSelect['endTR'] = foundationScoresSelect['endTR'] + 3


In [ ]:
# select only moral sentences
foundationScoresSelect_nonzero = foundationScoresSelect[foundationScoresSelect[whichFoundationScore] != 0]


In [ ]:
foundationScoresSelect_nonzero

In [ ]:

# extract the column as a numpy array
vals = foundationScoresSelect_nonzero["MAC_a_moral_nonmoral"].values

# compute all pairwise absolute differences
RDM_moralscores = np.abs(vals[:, None] - vals[None, :])

# if you prefer a DataFrame:
RDM_moralscores = pd.DataFrame(RDM_moralscores,
                               index=foundationScoresSelect_nonzero.index,
                               columns=foundationScoresSelect_nonzero.index)


In [ ]:

df = foundationScoresSelect_nonzero.copy()
E = len(df)
V, T, P = left.shape

# ------------------------------------------------------------
# 1. Extract event patterns
# ------------------------------------------------------------
def extract_patterns(hemisphere_array):
    V = hemisphere_array.shape[0]   # take voxel count from this hemisphere
    patterns = np.zeros((E, P, V))
    
    for e, (s, eTR) in enumerate(zip(df.startTR, df.endTR)):
        # average across TRs → (V, P), then transpose → (P, V)
        event_data = hemisphere_array[:, s:eTR+1, :].mean(axis=1).T
        patterns[e, :, :] = event_data

    return patterns

patterns_left  = extract_patterns(left)
patterns_right = extract_patterns(right)

# ------------------------------------------------------------
# 2. Compute leave-one-participant-out average patterns
# ------------------------------------------------------------
def compute_group_avg(patterns):
    E, P, V = patterns.shape
    avg = np.zeros((E, P, V))

    for p in range(P):
        others = np.delete(patterns, p, axis=1)  # shape = (E, P-1, V)
        avg[:, p, :] = others.mean(axis=1)
    return avg

avg_left  = compute_group_avg(patterns_left)
avg_right = compute_group_avg(patterns_right)

# ------------------------------------------------------------
# 3. Compute ISPC correlation RDMs
# ------------------------------------------------------------

# for every event pair (i, j), correlations are computed for every participant.

def compute_RDM(patterns, avg):
    E, P, V = patterns.shape
    RDM = np.zeros((E, E))

    for i in range(E):
        for j in range(E):
            corrs = []
            for p in range(P):
                x = patterns[i, p]
                y = avg[j, p]
                # correlation
                r = np.corrcoef(x, y)[0,1]
                corrs.append(r)
            RDM[i, j] = 1 - np.nanmean(corrs)  # dissimilarity
    return RDM

RDM_TPJ_left  = compute_RDM(patterns_left,  avg_left)
RDM_TPJ_right = compute_RDM(patterns_right, avg_right)



def compute_RDM_with_participant_corrs(patterns, avg):
    E, P, V = patterns.shape

    RDM = np.zeros((E, E))
    corrs_all = np.zeros((E, E, P))   # store each participant’s correlations

    for i in range(E):
        for j in range(E):
            for p in range(P):
                x = patterns[i, p]
                y = avg[j, p]
                r = np.corrcoef(x, y)[0,1]
                corrs_all[i, j, p] = r

            RDM[i, j] = 1 - np.nanmean(corrs_all[i, j, :])

    return RDM, corrs_all



# ------------------------------------------------------------
# 4. Correlate with BERT-score RDM
# ------------------------------------------------------------
# Extract upper triangle (excluding diagonal)
mask = np.triu(np.ones(RDM_df.shape), k=1).astype(bool)

bert_vec = RDM_df.values[mask]
left_vec  = RDM_TPJ_left[mask]
right_vec = RDM_TPJ_right[mask]

rho_left,  p_left  = spearmanr(bert_vec, left_vec)
rho_right, p_right = spearmanr(bert_vec, right_vec)

print("Left TPJ → behav RDM:")
print("Spearman rho =", rho_left, "; p =", p_left)

print("Right TPJ → behav RDM:")
print("Spearman rho =", rho_right, "; p =", p_right)


## run models hierarchically with BERT and moral scores to determine unique variance explained

In [ ]:
def compute_per_participant_RDMs(patterns):
    E, P, V = patterns.shape
    RDMs = np.zeros((P, E, E))

    for p in range(P):
        for i in range(E):
            for j in range(E):
                x = patterns[i, p]
                y = patterns[j, p]
                r = np.corrcoef(x, y)[0, 1]
                RDMs[p, i, j] = 1 - r  # dissimilarity
    return RDMs

In [ ]:
RDMs_pp_right = compute_per_participant_RDMs(patterns_right)



In [ ]:
RDMs_pp_left = compute_per_participant_RDMs(patterns_left)


## hierarchical models

In [ ]:
# ----------------------- USER SETTINGS -----------------------
n_splits = 10
random_state = 42
use_orthogonalize_global = True   # orthogonalize moral vs bert globally
n_perm = 1000
alpha = 0.05
clip_negative = True

# -------------------------------------------------------------

# -- STEPS TO MAKE SURE EVERYTHING IS CORRECT AS INPUT -----------------------
# make sure neural rdms are numpy array float32 values
# make sure diagonal is really zero (not some very very small value which happens sometimes
# -------------------------------------------------------------

def clean_rdm(mat, clip_negative=True):
    mat = np.array(mat, dtype=np.float32).copy()
    mat = (mat + mat.T) / 2.0
    np.fill_diagonal(mat, 0.0)
    if clip_negative:
        mat = np.clip(mat, 0.0, None)
    return mat

# Convert to numpy arrays if needed (sometimes I saved stuff as dataframes instead)
if isinstance(RDM_moralscores, pd.DataFrame):
    moral_arr_full = RDM_moralscores.values.copy()
else:
    moral_arr_full = np.array(RDM_moralscores).copy()

if isinstance(RDM_df, pd.DataFrame):
    bert_arr_full = RDM_df.values.copy()
else:
    bert_arr_full = np.array(RDM_df).copy()

RDMs_pp_right = np.array(RDMs_pp_right)  # shape (n_subjects, n_items, n_items)
n_subjects, n_items1, n_items2 = RDMs_pp_right.shape
n_items = n_items1
assert n_items1 == n_items2
assert moral_arr_full.shape == (n_items, n_items)
assert bert_arr_full.shape == (n_items, n_items)

bert_arr_full = clean_rdm(bert_arr_full)
moral_arr_full = clean_rdm(moral_arr_full)
RDMs_pp_right = np.array([clean_rdm(RDMs_pp_right[s]) for s in range(n_subjects)])

# masking for upper triangle
mask = np.triu(np.ones((n_items, n_items), dtype=bool), k=1)
bert_vec = bert_arr_full[mask].ravel()
moral_vec = moral_arr_full[mask].ravel()
assert bert_vec.shape[0] == moral_vec.shape[0] == mask.sum()

# orthogonalization
def orthogonalize(a, b):
    denom = np.dot(b, b)
    if denom == 0:
        return a.copy()
    proj = (np.dot(a, b) / denom) * b
    return a - proj

if use_orthogonalize_global:
    moral_vec = orthogonalize(moral_vec, bert_vec)

# ----------------------- functions used later -----------------------
def adjusted_r2(r2, n_obs, p):
    if n_obs - p - 1 <= 0:
        return np.nan
    return 1 - (1 - r2) * (n_obs - 1) / (n_obs - p - 1)

def fit_and_score(X_train, y_train, X_test, y_test):
    return LinearRegression().fit(X_train, y_train).score(X_test, y_test)

def bootstrap_ci(data, n_boot=5000, alpha=0.05, random_state=None):
    rng = np.random.RandomState(random_state)
    data = np.array(data)
    idx = rng.randint(0, len(data), size=(n_boot, len(data)))
    boot_means = np.nanmean(data[idx], axis=1)
    lo = np.percentile(boot_means, 100 * (alpha/2))
    hi = np.percentile(boot_means, 100 * (1-alpha/2))
    return lo, hi

# ----------------------- PER-SUBJECT CV & VARIANCE PARTITION -----------------------
kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# to save the results later
delta_R2_cv = np.zeros(n_subjects)
delta_adjR2_cv = np.zeros(n_subjects)
variance_partition = np.zeros((n_subjects, 3))
per_subject_fold_deltas = []


for s in range(n_subjects):

    # extract participant's neural RDM
    y_full = RDMs_pp_right[s][mask].ravel().astype(float)

    X_bert = bert_vec.reshape(-1,1)
    X_moral = moral_vec.reshape(-1,1)

    # split across bert only (reduced), moral only, full model (BERT and moral)
    R2_r_folds, R2_m_folds, R2_f_folds = [], [], []

    # 10 fold cross validation at feature level
    for train_idx, test_idx in kf.split(X_bert):
        Xr_train, Xr_test = X_bert[train_idx], X_bert[test_idx]
        Xm_train, Xm_test = X_moral[train_idx], X_moral[test_idx]
        y_train, y_test = y_full[train_idx], y_full[test_idx]

        # build the full matrix
        X_full_train = np.hstack([Xr_train, Xm_train])
        X_full_test  = np.hstack([Xr_test,  Xm_test])
        scaler = StandardScaler().fit(X_full_train)
        #standardize
        Xf_train_s = scaler.transform(X_full_train)
        Xf_test_s  = scaler.transform(X_full_test)

        Xr_train_s, Xr_test_s = Xf_train_s[:, [0]], Xf_test_s[:, [0]]
        Xm_train_s, Xm_test_s = Xf_train_s[:, [1]], Xf_test_s[:, [1]]
        Xf_train_s_final, Xf_test_s_final = Xf_train_s, Xf_test_s

        # fit the models and compute R2
        R2r = fit_and_score(Xr_train_s, y_train, Xr_test_s, y_test)
        R2m = fit_and_score(Xm_train_s, y_train, Xm_test_s, y_test)
        R2f = fit_and_score(Xf_train_s_final, y_train, Xf_test_s_final, y_test)

        # add to empty matrix
        R2_r_folds.append(R2r)
        R2_m_folds.append(R2m)
        R2_f_folds.append(R2f)

    R2_r_folds = np.array(R2_r_folds)
    R2_m_folds = np.array(R2_m_folds)
    R2_f_folds = np.array(R2_f_folds)

    #average across all folds
    R2r_mean = np.nanmean(R2_r_folds)
    R2m_mean = np.nanmean(R2_m_folds)
    R2f_mean = np.nanmean(R2_f_folds)

    #compute adjusted R2 per participant
    n_fold_obs = len(train_idx) + len(test_idx)
    adjR2r = np.nanmean([adjusted_r2(r, n_fold_obs, 1) for r in R2_r_folds])
    adjR2m = np.nanmean([adjusted_r2(r, n_fold_obs, 1) for r in R2_m_folds])
    adjR2f = np.nanmean([adjusted_r2(r, n_fold_obs, 2) for r in R2_f_folds])

    #compute delta R2 per participant
    delta_R2_cv[s] = R2f_mean - R2r_mean
    delta_adjR2_cv[s] = adjR2f - adjR2r

    # variance partitioning to isolate unique contribution of predictors (moral over bert)
    unique_B = R2f_mean - R2m_mean
    unique_M = R2f_mean - R2r_mean
    shared   = R2r_mean + R2m_mean - R2f_mean
    variance_partition[s, :] = [unique_B, unique_M, shared]

    # full minus bert (given that we want to know moral over and above bert)
    per_subject_fold_deltas.append(R2_f_folds - R2_r_folds)

# ----------------------- GROUP-LEVEL STATISTICS -----------------------
group_delta_mean = np.nanmean(delta_R2_cv)
group_delta_adj_mean = np.nanmean(delta_adjR2_cv)
t_stat, p_val_t = ttest_1samp(delta_R2_cv, 0, nan_policy='omit')
ci_lo, ci_hi = bootstrap_ci(delta_R2_cv, n_boot=2000, alpha=alpha, random_state=random_state)
vp_mean = variance_partition.mean(axis=0)

print("=== RESULTS ===")
print(f"Group mean ΔR2 (CV raw): {group_delta_mean:.6f}")
print(f"Group mean ΔAdjR2 (CV): {group_delta_adj_mean:.6f}")
print(f"T-test (ΔR2 across subjects): t={t_stat:.3f}, p={p_val_t:.5f}")
print(f"Bootstrap {100*(1-alpha):.1f}% CI for mean ΔR2: [{ci_lo:.6f}, {ci_hi:.6f}]")
print("Variance partitioning (mean across subjects):")
print(f"  unique_BERT: {vp_mean[0]:.6f}")
print(f"  unique_MORAL: {vp_mean[1]:.6f}")
print(f"  shared: {vp_mean[2]:.6f}")

# ----------------------- HIERARCHICAL PERMUTATION TEST -----------------------
rng = np.random.RandomState(random_state)
delta_perm = np.zeros(n_perm)

for p in trange(n_perm, desc="Permutations"):
    perm_idx = rng.permutation(n_items)
    moral_perm_vec = moral_arr_full[perm_idx][:, perm_idx][mask].ravel()
    if use_orthogonalize_global:
        moral_perm_vec = orthogonalize(moral_perm_vec, bert_vec)

    subj_deltas = np.zeros(n_subjects)
    for s in range(n_subjects):
        y_full = RDMs_pp_right[s][mask].ravel().astype(float)

        Xr = bert_vec.reshape(-1,1)
        Xm_perm = moral_perm_vec.reshape(-1,1)

        R2r_f, R2f_f = [], []
        for train_idx, test_idx in kf.split(Xr):
            Xr_train, Xr_test = Xr[train_idx], Xr[test_idx]
            Xm_train, Xm_test = Xm_perm[train_idx], Xm_perm[test_idx]
            y_train, y_test = y_full[train_idx], y_full[test_idx]

            scaler = StandardScaler().fit(np.hstack([Xr_train, Xm_train]))
            Xf_train_s = scaler.transform(np.hstack([Xr_train, Xm_train]))
            Xf_test_s  = scaler.transform(np.hstack([Xr_test, Xm_test]))

            Xr_train_s = Xf_train_s[:, [0]]; Xr_test_s = Xf_test_s[:, [0]]
            Xf_train_final, Xf_test_final = Xf_train_s, Xf_test_s

            R2r_f.append(fit_and_score(Xr_train_s, y_train, Xr_test_s, y_test))
            R2f_f.append(fit_and_score(Xf_train_final, y_train, Xf_test_final, y_test))

        subj_deltas[s] = np.nanmean(R2f_f) - np.nanmean(R2r_f)

    delta_perm[p] = np.nanmean(subj_deltas)

p_value_perm = np.mean(delta_perm >= group_delta_mean)
print(f"Permutation p-value (one-sided ΔR2 > 0): {p_value_perm:.5f}")

# ----------------------- RESULTS OBJECT -----------------------
results = {
    "delta_R2_cv": delta_R2_cv,
    "delta_adjR2_cv": delta_adjR2_cv,
    "variance_partition": variance_partition,
    "per_subject_fold_deltas": per_subject_fold_deltas,
    "group_delta_mean": group_delta_mean,
    "group_delta_adj_mean": group_delta_adj_mean,
    "t_stat": t_stat,
    "p_val_t": p_val_t,
    "bootstrap_ci": (ci_lo, ci_hi),
    "perm_null": delta_perm,
    "perm_pvalue": p_value_perm
}


In [ ]:

def plot_rsa_2panel_inset(delta_R2_cv, variance_partition, perm_pvalue, n_subjects, ylim_factor=1.5):
    """
    Left panel: ΔR² across subjects (violin + mean ± SEM)
    Right panel: Variance partitioning (mean ± SEM) with zoomed inset for small bars
    """
    sns.set(style="whitegrid")
    fig, axes = plt.subplots(1, 2, figsize=(14,6))

    # ------------------ Left panel: ΔR² ------------------
    ax = axes[0]
    sns.violinplot(y=delta_R2_cv, inner=None, color='skyblue', ax=ax)
    mean_delta = np.mean(delta_R2_cv)
    sem_delta = np.std(delta_R2_cv, ddof=1)/np.sqrt(n_subjects)
    ax.errorbar(0, mean_delta, yerr=sem_delta, fmt='o', color='darkred', capsize=5, label='Mean ± SEM')
    ax.axhline(0, linestyle='--', color='red')
    ax.set_ylabel("ΔR² (BERT → BERT+MORAL)")
    ax.set_title("ΔR² across subjects")
    y_min, y_max = ax.get_ylim()
    y_center = (y_min + y_max) / 2
    y_range = (y_max - y_min) * ylim_factor / 2
    ax.set_ylim(y_center - y_range, y_center + y_range)
    ax.text(0, ax.get_ylim()[1]*0.9, f"Permutation p = {perm_pvalue:.3f}", ha='center', color='darkred', fontsize=11)

    # ------------------ Right panel: Variance partitioning ------------------
    ax = axes[1]
    labels = ["unique BERT", "unique MORAL", "shared"]
    vp_mean = variance_partition.mean(axis=0)
    vp_sem = variance_partition.std(axis=0)/np.sqrt(n_subjects)
    bar_colors = ["lightgray","skyblue","green"]
    
    # Main bars
    ax.bar(labels, vp_mean, yerr=vp_sem, capsize=5, color=bar_colors, edgecolor='black', linewidth=1.5, zorder=2)
    ax.set_ylabel("R² contribution")
    ax.set_title("Variance partitioning (mean ± SEM)")

    # ------------------ Zoomed inset for small bars ------------------
    # inset_axes(parent_ax, [x0, y0, width, height])
    ax_inset = inset_axes(ax, width="50%", height="50%", loc='upper right', borderpad=2)
    ax_inset.bar(labels, vp_mean, yerr=vp_sem, capsize=5, color=bar_colors, edgecolor='black', linewidth=1.5, zorder=2)
    # zoom to lower values
    tiny_max = max(vp_mean[1:] + vp_sem[1:]) * 1.3
    ax_inset.set_ylim(0, tiny_max)
    ax_inset.set_title("Zoomed for small contributions", fontsize=10)
    ax_inset.tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig(str(save_dir)+'/'+f"RSA_combinedModelBERTandMoral_{whichROI}_{whichStory}_{whichVersion}")
    plt.show()


In [ ]:

plot_rsa_2panel_inset(
    delta_R2_cv=results['delta_R2_cv'],          # array of ΔR² per subject
    variance_partition=results['variance_partition'],  # shape (n_subjects, 3)
    perm_pvalue=results['perm_pvalue'],          # permutation p-value
    n_subjects=n_subjects,                       # number of subjects
    ylim_factor=1.5                              # optional, expands left panel y-axis
)

